# Parquet -> GR00T observation

Read a recorded Station `.parquet` with pandas and turn each row into the
`{"video", "state", "language"}` dict the GR00T policy server expects
(see `DATA_FORMAT.md`). Last cell optionally sends one frame to the server.

In [1]:
import io

import numpy as np
import pandas as pd
from PIL import Image

PARQUET = "../dataset-cheese1.parquet"
IMAGE_KEYS = ("cam0", "cam1", "cam2")  # must match the checkpoint's modality config, in Frame image order
STATE_KEY = "state"


In [2]:
df = pd.read_parquet(PARQUET, engine="pyarrow")
print(df.dtypes, len(df), sep="\n")
df.head()

episode_start_ns                    uint64
global_frame_id                     object
timestamp_ns_since_episode_start    uint64
joints                              object
images                              object
task                                object
dtype: object
408


,episode_start_ns,global_frame_id,timestamp_ns_since_episode_start,joints,images,task
0,1792873207537838867,b'\xa5]\xfe\x01',0,"[{'range_min': 1109, 'range_max': 2987, 'posit...",[{'jpeg': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,put the cheese on pizza
1,1792873207537838867,b'+^\xfe\x01',100317680,"[{'range_min': 1109, 'range_max': 2987, 'posit...",[{'jpeg': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,put the cheese on pizza
2,1792873207537838867,b'\xdc^\xfe\x01',200349693,"[{'range_min': 1109, 'range_max': 2987, 'posit...",[{'jpeg': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,put the cheese on pizza
3,1792873207537838867,b'\xa5_\xfe\x01',300782322,"[{'range_min': 1109, 'range_max': 2987, 'posit...",[{'jpeg': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,put the cheese on pizza
4,1792873207537838867,b'_`\xfe\x01',400965717,"[{'range_min': 1109, 'range_max': 2987, 'posit...",[{'jpeg': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,put the cheese on pizza


In [3]:
row = df.iloc[0]
print("joints:", len(row.joints), "images:", len(row.images), "task:", row.task)
pd.DataFrame(list(row.joints))

joints: 8 images: 3 task: put the cheese on pizza


,range_min,range_max,position,position_norm,goal,goal_norm,current_ma,velocity
0,1109,2987,1896,0.419063,1845,0.391906,1,0
1,991,3105,1036,0.021287,1039,0.022706,0,0
2,902,3194,1194,0.127400,1182,0.122164,2,0
3,921,3175,3154,0.990683,3155,0.991127,0,0
4,138,3958,3784,0.954450,3779,0.953141,0,0
5,1036,3060,1157,0.059783,1157,0.059783,0,0
6,146,3950,172,0.006835,166,0.005258,1,0
7,1279,2817,1659,0.247074,1655,0.244473,0,0


In [4]:
def decode_jpeg(b: bytes) -> np.ndarray:
    with Image.open(io.BytesIO(b)) as im:
        return np.asarray(im.convert("RGB"), dtype=np.uint8)


def row_to_observation(row, image_keys=IMAGE_KEYS, state_key=STATE_KEY, task=None) -> dict:
    """One parquet row -> GR00T obs dict, shapes (B=1, T=1, ...)."""
    video = {k: decode_jpeg(row.images[i]["jpeg"])[None, None] for i, k in enumerate(image_keys)}
    state = {state_key: np.array([[[j["position_norm"] for j in row.joints]]], dtype=np.float32)}
    language = {"task": [[task or row.task]]}
    for k, v in video.items():
        assert isinstance(v, np.ndarray) and v.dtype == np.uint8 and v.ndim == 5, f"video[{k!r}]: {type(v)}"
    return {"video": video, "state": state, "language": language}


obs = row_to_observation(df.iloc[0])
{m: {k: getattr(v, "shape", v) for k, v in d.items()} for m, d in obs.items()}

{'video': {'cam0': (1, 1, 224, 224, 3),
  'cam1': (1, 1, 224, 224, 3),
  'cam2': (1, 1, 224, 224, 3)},
 'state': {'state': (1, 1, 8)},
 'language': {'task': [['put the cheese on pizza']]}}

In [5]:
obs["video"]["cam0"]
# vizualize the video


array([[[[[  0,   0,   2],
          [  0,   0,   2],
          [  0,   0,   2],
          ...,
          [ 62,  63,  68],
          [ 40,  41,  46],
          [ 18,  19,  23]],

         [[  0,   0,   2],
          [  0,   0,   2],
          [  0,   0,   2],
          ...,
          [ 12,  13,  17],
          [ 11,  12,  16],
          [ 18,  19,  23]],

         [[  0,   0,   2],
          [  0,   0,   2],
          [  0,   0,   2],
          ...,
          [  5,   6,   8],
          [ 10,  12,  11],
          [  7,  11,  10]],

         ...,

         [[ 76,  77,  72],
          [ 72,  73,  65],
          [ 78,  78,  66],
          ...,
          [102, 112, 114],
          [106, 111, 117],
          [102, 105, 112]],

         [[ 72,  72,  70],
          [ 72,  71,  66],
          [ 73,  73,  63],
          ...,
          [ 97, 109, 109],
          [102, 110, 113],
          [104, 109, 113]],

         [[ 68,  66,  67],
          [ 71,  70,  66],
          [ 73,  70,  61],
         

In [6]:
# Optional: send to the policy server (needs `uv pip install -e .` from the repo root)
from inference_client.policy_client import PolicyClient

POLICY_HOST, POLICY_PORT = "81.85.2.138", 5556

with PolicyClient(host=POLICY_HOST, port=POLICY_PORT, timeout_ms=5000) as policy:
    assert policy.ping(), "can't reach policy server"
    print("modality config:", policy.get_modality_config())
    action, info = policy.get_action(obs)
    print({k: v.shape for k, v in action.items()}, info)

modality config: {'video': {'__ModalityConfig_class__': True, 'as_json': {'delta_indices': [0], 'modality_keys': ['cam0', 'cam1', 'cam2'], 'sin_cos_embedding_keys': None, 'mean_std_embedding_keys': None, 'action_configs': None}}, 'state': {'__ModalityConfig_class__': True, 'as_json': {'delta_indices': [0], 'modality_keys': ['state'], 'sin_cos_embedding_keys': None, 'mean_std_embedding_keys': None, 'action_configs': None}}, 'action': {'__ModalityConfig_class__': True, 'as_json': {'delta_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], 'modality_keys': ['action'], 'sin_cos_embedding_keys': None, 'mean_std_embedding_keys': None, 'action_configs': [{'rep': 'ABSOLUTE', 'type': 'NON_EEF', 'format': 'DEFAULT', 'state_key': None}]}}, 'language': {'__ModalityConfig_class__': True, 'as_json': {'delta_indices': [0], 'modality_keys': ['task'], 'sin_cos_embedding_keys': None, 'mean_std_embedding_keys': None, 'action_configs': None}}}


RuntimeError: Server error: Video key 'cam0' must be a numpy array. Got <class 'dict'>

In [ ]:
type(obs["video"]["cam0"])

numpy.ndarray